# Part 3 — Machine Learning: Predict No-Shows

Tasks 3.1–3.5. The prediction score is produced on the evening before the appointment, after reminders have been sent or skipped. The test set is for final reporting only: model and threshold selection use training-only cross-validation.

Load `appointments_clean.csv` without shuffling or re-sorting its rows.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

import os
import subprocess
REPO_URL = 'https://github.com/Subham9339063436/CareLoop-Clinics-No-Show-Guard.git'
REPO_DIR = Path('/content/CareLoop-Clinics-No-Show-Guard')
if not (Path.cwd() / 'appointments_clean.csv').exists():
    if not (REPO_DIR / 'appointments_clean.csv').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

ROOT = Path.cwd()
df = pd.read_csv(ROOT / 'appointments_clean.csv')
assert df.shape == (990, 16), 'Run Part 1 and verify its 990-row clean output first.'
appointment_ids = df['appointment_id'].copy()


## Task 3.1 — Features, split and baseline

In [2]:
X = df[[
    'age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows',
    'reminder_sent', 'booking_channel', 'visit_type'
]].copy()
X['reminder_sent'] = X['reminder_sent'].map({'Yes': 1, 'No': 0})
X = pd.get_dummies(X, columns=['booking_channel', 'visit_type'])
y = df['no_show'].copy()
assert X.shape[1] == 11, f'Expected 11 features, received {X.shape[1]}'

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
test_appointment_ids = appointment_ids.loc[X_test.index]
baseline_accuracy = (y_test == 0).mean()
print('Features:', X.shape[1], list(X.columns))
print('Training rows:', len(X_train), 'Test rows:', len(X_test))
print('No-shows in test set:', int(y_test.sum()))
print('Always-Attended baseline accuracy:', baseline_accuracy)


Features: 11 ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Training rows: 792 Test rows: 198
No-shows in test set: 49
Always-Attended baseline accuracy: 0.7525252525252525


## Task 3.2 — Logistic regression

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
logistic = LogisticRegression(max_iter=1000)
logistic.fit(X_train_scaled, y_train)
logistic_test_prob = logistic.predict_proba(X_test_scaled)[:, 1]
logistic_test_pred = logistic.predict(X_test_scaled)

logistic_metrics = {
    'accuracy': accuracy_score(y_test, logistic_test_pred),
    'precision': precision_score(y_test, logistic_test_pred, zero_division=0),
    'recall': recall_score(y_test, logistic_test_pred, zero_division=0),
    'f1': f1_score(y_test, logistic_test_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, logistic_test_prob),
    'confusion_matrix': confusion_matrix(y_test, logistic_test_pred).tolist()
}
print('Logistic regression test metrics:', logistic_metrics)
print('Coefficients in feature order:')
for feature, coefficient in zip(X.columns, logistic.coef_[0]):
    print(feature, coefficient)

reminder_index = list(X.columns).index('reminder_sent')
reminder_coefficient = logistic.coef_[0][reminder_index]
reminder_sd = scaler.scale_[reminder_index]
print('exp(scaled reminder coefficient):', np.exp(reminder_coefficient))
print('Reminder feature standard deviation:', reminder_sd)
print('exp(coefficient / feature SD), No-to-Yes effect:', np.exp(reminder_coefficient / reminder_sd))
print('Interpretation: in this model, holding other features constant, reminder_sent=Yes is associated with lower no-show odds; this observational association does not prove reminders cause the reduction.')


Logistic regression test metrics: {'accuracy': 0.7878787878787878, 'precision': 0.6296296296296297, 'recall': 0.3469387755102041, 'f1': 0.4473684210526316, 'roc_auc': 0.788933022873579, 'confusion_matrix': [[139, 10], [32, 17]]}
Coefficients in feature order:
age -0.2538801442716842
distance_km 0.25344770956317536
fee_inr 0.4527614630856424
lead_days 0.6144707304969084
prior_no_shows 0.6496116219621333
reminder_sent -0.5806686033659922
booking_channel_App -0.04680357040738988
booking_channel_Phone 0.21588137492827028
booking_channel_Walk-In -0.18636416836300687
visit_type_Consultation 0.015658011798355062
visit_type_Follow-up -0.015658011798355315
exp(scaled reminder coefficient): 0.559524141750829
Reminder feature standard deviation: 0.4949752621616814
exp(coefficient / feature SD), No-to-Yes effect: 0.3093980903124433
Interpretation: in this model, holding other features constant, reminder_sent=Yes is associated with lower no-show odds; this observational association does not prove r

## Task 3.3 — Decision trees and overfitting

In [4]:
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train, y_train)
depth3_tree = DecisionTreeClassifier(random_state=42, max_depth=3)
depth3_tree.fit(X_train, y_train)
for label, model in [('Full-depth tree', full_tree), ('Depth-3 tree', depth3_tree)]:
    print(label,
          'depth=', model.get_depth(),
          'training accuracy=', accuracy_score(y_train, model.predict(X_train)),
          'test accuracy=', accuracy_score(y_test, model.predict(X_test)))
print('A large train/test gap in the full-depth tree indicates overfitting; the depth-3 limit reduces memorisation.')


Full-depth tree depth= 15 training accuracy= 1.0 test accuracy= 0.6717171717171717
Depth-3 tree depth= 3 training accuracy= 0.821969696969697 test accuracy= 0.7777777777777778
A large train/test gap in the full-depth tree indicates overfitting; the depth-3 limit reduces memorisation.


## Task 3.4 — Random forest and model comparison

In [5]:
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest.fit(X_train, y_train)
rf_test_prob = random_forest.predict_proba(X_test)[:, 1]
rf_test_pred = random_forest.predict(X_test)
rf_metrics = {
    'accuracy': accuracy_score(y_test, rf_test_pred),
    'recall': recall_score(y_test, rf_test_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, rf_test_prob),
    'training_accuracy': accuracy_score(y_train, random_forest.predict(X_train))
}
print('Random forest metrics:', rf_metrics)
feature_importance = pd.Series(random_forest.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Top three feature importances:', feature_importance.head(3))

comparison = pd.DataFrame([
    {'model': 'Logistic regression', 'accuracy': logistic_metrics['accuracy'], 'recall': logistic_metrics['recall'], 'roc_auc': logistic_metrics['roc_auc']},
    {'model': 'Decision tree (depth 3)', 'accuracy': accuracy_score(y_test, depth3_tree.predict(X_test)), 'recall': recall_score(y_test, depth3_tree.predict(X_test), zero_division=0), 'roc_auc': roc_auc_score(y_test, depth3_tree.predict_proba(X_test)[:, 1])},
    {'model': 'Random forest (100 trees)', 'accuracy': rf_metrics['accuracy'], 'recall': rf_metrics['recall'], 'roc_auc': rf_metrics['roc_auc']}
])
print(comparison.to_string(index=False))


Random forest metrics: {'accuracy': 0.7474747474747475, 'recall': 0.30612244897959184, 'roc_auc': 0.7057252431173813, 'training_accuracy': 1.0}
Top three feature importances: lead_days      0.219905
age            0.196788
distance_km    0.194739
dtype: float64
                    model  accuracy   recall  roc_auc
      Logistic regression  0.787879 0.346939 0.788933
  Decision tree (depth 3)  0.777778 0.224490 0.699699
Random forest (100 trees)  0.747475 0.306122 0.705725


## Task 3.5 — Training-only CV selection, OOF threshold, one test evaluation

In [6]:
# (a) Tune the random forest using training data only.
param_grid = {'n_estimators': [100, 200], 'max_depth': [3, 5, 8]}
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42), param_grid,
    cv=5, scoring='roc_auc'
)
rf_grid.fit(X_train, y_train)
print('Tuned forest best parameters:', rf_grid.best_params_)
print('Tuned forest best CV ROC-AUC:', rf_grid.best_score_)

# (b) Compare all candidate models using training-only cross-validation.
# The LR pipeline refits StandardScaler inside each CV fold.
logistic_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])
cv_candidates = {
    'Logistic regression': logistic_pipeline,
    'Decision tree depth 3': DecisionTreeClassifier(random_state=42, max_depth=3),
    'Random forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Tuned random forest': rf_grid.best_estimator_
}
cv_auc = {
    name: cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc').mean()
    for name, model in cv_candidates.items()
}
print('Training CV ROC-AUC:', cv_auc)
winning_name = max(cv_auc, key=cv_auc.get)
winning_model = cv_candidates[winning_name]
print('Selected model (highest training CV ROC-AUC):', winning_name)

# (c) Threshold choice is based only on out-of-fold training probabilities.
oof_prob = cross_val_predict(
    winning_model, X_train, y_train, cv=5, method='predict_proba'
)[:, 1]
threshold_results = []
for threshold in [0.5, 0.35]:
    flags = (oof_prob >= threshold).astype(int)
    threshold_results.append({
        'threshold': threshold,
        'precision': precision_score(y_train, flags, zero_division=0),
        'recall': recall_score(y_train, flags, zero_division=0),
        'f1': f1_score(y_train, flags, zero_division=0),
        'flagged': int(flags.sum())
    })
threshold_table = pd.DataFrame(threshold_results)
print(threshold_table.to_string(index=False))
chosen_threshold = float(threshold_table.sort_values('f1', ascending=False).iloc[0]['threshold'])
print('Chosen threshold (higher training OOF F1):', chosen_threshold)
print('Recall matters because a missed warning may leave a doctor slot empty, while an extra follow-up call costs staff time.')

# (d) Fit the winner on all training rows and evaluate the test set once.
winning_model.fit(X_train, y_train)
test_risk_prob = winning_model.predict_proba(X_test)[:, 1]
test_flag = (test_risk_prob >= chosen_threshold).astype(int)
final_test_metrics = {
    'precision': precision_score(y_test, test_flag, zero_division=0),
    'recall': recall_score(y_test, test_flag, zero_division=0),
    'flagged': int(test_flag.sum()),
    'roc_auc': roc_auc_score(y_test, test_risk_prob)
}
print('Final held-out test metrics:', final_test_metrics)
print('Clinic threshold should be set using the number of follow-up calls the front desk can make in an evening against the cost of an empty slot.')

# (e) Export one risk score for every held-out test row.
test_risk_scores = pd.DataFrame({
    'appointment_id': test_appointment_ids.values,
    'no_show': y_test.values,
    'risk_prob': test_risk_prob,
    'flag': test_flag
})
test_risk_scores.to_csv(ROOT / 'test_risk_scores.csv', index=False)
print('Saved test_risk_scores.csv:', test_risk_scores.shape,
      '| flagged:', int(test_risk_scores['flag'].sum()))


Tuned forest best parameters: {'max_depth': 3, 'n_estimators': 200}
Tuned forest best CV ROC-AUC: 0.7673242251739405


Training CV ROC-AUC: {'Logistic regression': np.float64(0.7737139899948249), 'Decision tree depth 3': np.float64(0.6784494968661952), 'Random forest': np.float64(0.7466024782933701), 'Tuned random forest': np.float64(0.7673242251739405)}
Selected model (highest training CV ROC-AUC): Logistic regression
 threshold  precision   recall       f1  flagged
      0.50   0.575342 0.248521 0.347107       73
      0.35   0.512987 0.467456 0.489164      154
Chosen threshold (higher training OOF F1): 0.35
Recall matters because a missed warning may leave a doctor slot empty, while an extra follow-up call costs staff time.
Final held-out test metrics: {'precision': 0.5714285714285714, 'recall': 0.5714285714285714, 'flagged': 49, 'roc_auc': 0.788933022873579}
Clinic threshold should be set using the number of follow-up calls the front desk can make in an evening against the cost of an empty slot.
Saved test_risk_scores.csv: (198, 4) | flagged: 49


## Required limits to report in the final recommendation

The medians, quartiles and outlier fences in Part 1 were computed on all 990 rows before splitting, so the test rows had a small indirect influence on cleaning. A patient can have multiple bookings, so the same patient may appear in both train and test. The random split also does not simulate a real past-to-future deployment.